# Phase S1: Cooling Theory Validation
## ThermoRG-NN Project

**Goal**: Validate BatchNorm and Skip connection as "cooling mechanisms"

### Theory
- β(J, γ) = β₀(J) · φ(γ)
- E(J, γ) = E₀(J) · exp(-κγ)
- φ(γ) = (γ_c/(γ_c+γ))·exp(-γ/γ_c)

### Predictions
- BatchNorm should reduce β by φ ≈ 0.66
- Skip should reduce β by φ ≈ 0.93-0.98

In [ ]:
# Setup
import os, json, math, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import linalg
import warnings
warnings.filterwarnings('ignore')

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Clone repo if needed
if not os.path.exists('ThermoRG-NN'):
    !git clone https://github.com/xliu203/ThermoRG-NN.git
    %cd ThermoRG-NN

os.chdir('ThermoRG-NN')

## Configuration

In [ ]:
# 6 configurations
CONFIGS = [
    ('None_NoSkip',  'none',       False),
    ('LN_NoSkip',    'layernorm',  False),
    ('BN_NoSkip',    'batchnorm',  False),
    ('None_Skip',   'none',       True),
    ('LN_Skip',      'layernorm',  True),
    ('BN_Skip',      'batchnorm',  True),
]

# D values via base channel width
D_VALUES = [32, 48, 64, 96]
SEEDS = [42, 123]
EPOCHS = 200
BATCH_SIZE = 128
LR = 0.01
WD = 5e-4
MOMENTUM = 0.9

OUT_DIR = Path('./phase_s1_results')
OUT_DIR.mkdir(exist_ok=True)
print(f"Configs: {len(CONFIGS)}, D values: {D_VALUES}, Epochs: {EPOCHS}")

## Network Definition

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, norm_type='none', use_skip=False, stride=1):
        super().__init__()
        self.norm_type = norm_type
        self.use_skip = use_skip
        self.conv = nn.Conv2d(in_ch, out_ch, 3, padding=1, stride=stride, bias=False)
        if norm_type == 'layernorm':
            self.norm = nn.LayerNorm([out_ch, 32, 32])
        elif norm_type == 'batchnorm':
            self.norm = nn.BatchNorm2d(out_ch)
        else:
            self.norm = nn.Identity()
        self.act = nn.GELU()
        if use_skip and in_ch == out_ch and stride == 1:
            self.skip = nn.Identity()
        elif use_skip:
            self.skip = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )
        else:
            self.skip = None

    def forward(self, x):
        out = self.norm(self.conv(x))
        out = self.act(out)
        if self.use_skip and self.skip is not None:
            out = out + self.skip(x)
        return out


class ValidationNet(nn.Module):
    def __init__(self, base_ch=64, norm_type='none', use_skip=False, n_classes=10):
        super().__init__()
        channels = [3, base_ch, base_ch*2, base_ch*2]
        self.blocks = nn.ModuleList()
        for i in range(len(channels) - 1):
            self.blocks.append(ConvBlock(
                channels[i], channels[i+1],
                norm_type=norm_type,
                use_skip=(i > 0 and use_skip),
                stride=1
            ))
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(channels[-1], n_classes)

    def get_conv_weights(self):
        return [m.weight.data for m in self.modules() if isinstance(m, nn.Conv2d)]

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

print("Network defined.")

## J_topo Computation

In [ ]:
def compute_D_eff(W):
    if W.dim() == 4:
        W = W.view(W.size(0), -1)
    fro_sq = (W ** 2).sum().item()
    S = linalg.svd(W.to('cpu')).S
    spec_sq = S[0].item() ** 2 + 1e-12
    return fro_sq / spec_sq

def compute_J_topo(weights, d_input=3.0):
    eta_vals = []
    d_prev = d_input
    for W in weights:
        if W.dim() == 4:
            W = W.view(W.size(0), -1)
        D_eff = compute_D_eff(W)
        eta = D_eff / max(d_prev, 1e-8)
        eta_vals.append(max(eta, 1e-8))
        d_prev = D_eff
    L = len(eta_vals)
    log_sum = sum(abs(math.log(e)) for e in eta_vals)
    return math.exp(-log_sum / L) if L > 0 else 0.0

print("J_topo functions defined.")

## Data Loading

In [ ]:
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader

transform_train = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize([0.4914, 0.4822, 0.4465], [0.2470, 0.2435, 0.2616]),
])
transform_val = T.Compose([
    T.ToTensor(),
    T.Normalize([0.4914, 0.4822, 0.4465], [0.2470, 0.2435, 0.2616]),
])

train_ds = CIFAR10(root='./data', train=True, transform=transform_train, download=True)
val_ds   = CIFAR10(root='./data', train=False, transform=transform_val, download=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

## Training Functions

In [ ]:
from scipy.optimize import curve_fit

def train_model(model, train_loader, val_loader, epochs, lr, wd, momentum, device):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    model = model.to(device)

    best_val_loss = float('inf')
    best_val_acc = 0
    best_epoch = 0

    for epoch in range(epochs):
        model.train()
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()
        scheduler.step()

        model.eval()
        val_loss = 0
        correct = 0
        total = 0
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                out = model(X)
                val_loss += criterion(out, y).item() * X.size(0)
                correct += (out.argmax(1) == y).sum().item()
                total += X.size(0)

        val_loss /= total
        val_acc = correct / total

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_acc = val_acc
            best_epoch = epoch + 1

    return {
        'best_val_loss': best_val_loss,
        'best_val_acc': best_val_acc,
        'best_epoch': best_epoch,
    }


def fit_scaling_law(losses_by_d, d_values):
    def power_law(D, alpha, beta, E):
        return alpha * np.array(D) ** (-beta) + E

    Ds = np.array(d_values)
    losses = np.array([losses_by_d[d] for d in d_values])

    try:
        popt, _ = curve_fit(power_law, Ds, losses, p0=[10.0, 0.5, 0.5],
                           bounds=([0, 0, 0], [1000, 5, 10]), maxfev=10000)
        alpha, beta, E = popt
        preds = power_law(Ds, alpha, beta, E)
        ss_res = ((losses - preds) ** 2).sum()
        ss_tot = ((losses - losses.mean()) ** 2).sum()
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
        return {'alpha': float(alpha), 'beta': float(beta), 'E': float(E), 'R2': float(r2)}
    except Exception as e:
        return {'alpha': None, 'beta': None, 'E': None, 'R2': 0.0, 'error': str(e)}

print("Training functions defined.")

## Main Experiment Loop

In [ ]:
from datetime import datetime

all_results = {
    'timestamp': datetime.now().isoformat(),
    'config': {'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'lr': LR, 'wd': WD,
               'd_values': D_VALUES, 'seeds': SEEDS},
    'configs': []
}

total_start = time.time()

for config_name, norm_type, use_skip in CONFIGS:
    print(f"\n{'='*60}")
    print(f"[{config_name}] norm={norm_type}, skip={use_skip}")
    print(f"{'='*60}")
    
    cfg_start = time.time()
    cfg_result = {'name': config_name, 'norm': norm_type, 'skip': use_skip, 'D_results': {}}

    # J_topo at init
    model_init = ValidationNet(base_ch=64, norm_type=norm_type, use_skip=use_skip)
    weights_init = model_init.get_conv_weights()
    J_topo_init = compute_J_topo(weights_init)
    cfg_result['J_topo_init'] = J_topo_init
    print(f"J_topo(init) = {J_topo_init:.4f}")
    del model_init; torch.cuda.empty_cache()

    # Train for each D
    for base_ch in D_VALUES:
        print(f"  base_ch={base_ch}...", end=' ', flush=True)
        d_result = {'base_ch': base_ch, 'seeds': {}}
        losses = []

        for seed in SEEDS:
            torch.manual_seed(seed)
            torch.cuda.manual_seed(seed)
            
            model = ValidationNet(base_ch=base_ch, norm_type=norm_type, use_skip=use_skip)
            result = train_model(model, train_loader, val_loader, EPOCHS, LR, WD, MOMENTUM, device)
            
            d_result['seeds'][seed] = result
            losses.append(result['best_val_loss'])
            print('.', end='', flush=True)
            del model; torch.cuda.empty_cache()

        d_result['avg_val_loss'] = float(np.mean(losses))
        print(f" avg={d_result['avg_val_loss']:.4f}")
        cfg_result['D_results'][str(base_ch)] = d_result

    # Fit scaling law
    losses_by_d = {str(ch): cfg_result['D_results'][str(ch)]['avg_val_loss'] for ch in D_VALUES}
    fit = fit_scaling_law(losses_by_d, D_VALUES)
    cfg_result['scaling_fit'] = fit
    print(f"  Fit: α={fit['alpha']:.2f}, β={fit['beta']:.4f}, E={fit['E']:.4f}, R²={fit['R2']:.4f}")

    cfg_time = (time.time() - cfg_start) / 60
    cfg_result['wall_time_min'] = cfg_time
    print(f"  Time: {cfg_time:.1f} min")
    
    all_results['configs'].append(cfg_result)

total_time = (time.time() - total_start) / 60
all_results['total_time_min'] = total_time
print(f"\nTotal runtime: {total_time:.1f} min")

## Save Results

In [ ]:
# Save
out_file = OUT_DIR / 'phase_s1_results.json'
with open(out_file, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f"Saved to {out_file}")

## Summary & Analysis

In [ ]:
# Summary table
baseline = next(c for c in all_results['configs'] if c['name'] == 'None_NoSkip')
base_beta = baseline['scaling_fit']['beta'] or 0.4

print(f"\n{'Config':<15} {'J_topo':<8} {'Beta':<10} {'φ(β)':<10} {'E':<10}")
print("-" * 60)
for cfg in all_results['configs']:
    fit = cfg['scaling_fit']
    beta = fit['beta'] or 0
    phi = beta / base_beta if base_beta > 0 else 0
    print(f"{cfg['name']:<15} {cfg['J_topo_init']:<8.4f} {beta:<10.4f} {phi:<10.3f} {fit['E']:<10.4f}")

# Cooling analysis
print("\n" + "="*60)
print("COOLING ANALYSIS")
print("="*60)

def get_cfg(name):
    return next(c for c in all_results['configs'] if c['name'] == name)

bn = get_cfg('BN_NoSkip')
ln = get_cfg('LN_NoSkip')
none = get_cfg('None_NoSkip')
skip_ln = get_cfg('LN_Skip')
noskip_ln = get_cfg('LN_NoSkip')

beta_none = none['scaling_fit']['beta'] or 0.4
beta_ln = ln['scaling_fit']['beta'] or 0.4
beta_bn = bn['scaling_fit']['beta'] or 0.4

print(f"\n1. Normalization Cooling:")
print(f"   None vs LN: φ = {beta_ln/beta_none:.3f}")
print(f"   None vs BN: φ = {beta_bn/beta_none:.3f}")
print(f"   LN vs BN:   φ = {beta_bn/beta_ln:.3f}")
print(f"   Prediction: φ_BN ≈ 0.66")

print(f"\n2. Skip Cooling (LN baseline):")
beta_skip = skip_ln['scaling_fit']['beta'] or 0.4
beta_noskip = noskip_ln['scaling_fit']['beta'] or 0.4
print(f"   Skip/NoSkip: φ = {beta_skip/beta_noskip:.3f}")
print(f"   Prediction: φ_skip ≈ 0.93-0.98")

print(f"\n3. Combined (BN+Skip):")
bn_skip = get_cfg('BN_Skip')
beta_bnskip = bn_skip['scaling_fit']['beta'] or 0.4
phi_combined = beta_bnskip / beta_none
phi_add = (beta_bn/beta_none) * (beta_skip/beta_noskip)
print(f"   φ_combined = {phi_combined:.3f}")
print(f"   φ_additivity = φ_BN × φ_skip = {phi_add:.3f}")